# WhatsApp Digital Twin - Qwen 2.5 7B Instruct Fine-Tuning
**Architecture:** QLoRA 4-bit (Unsloth) + GGUF Quantization for Ollama
**Hardware requirement:** Kaggle GPU T4 x2 or P100 (Turn Internet ON in Notebook Settings)

In [ ]:
# Cell 1: Install Unsloth & Fast Training Dependencies
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.29" trl peft accelerate bitsandbytes
print("Unsloth and dependencies installed successfully!")

In [ ]:
# Cell 2: Locate Dataset in /kaggle/input/
import os

def find_dataset_file(filename="train.jsonl"):
    if os.path.exists(filename):
        return os.path.abspath(filename)
    if os.path.exists("/kaggle/input"):
        for root, _, files in os.walk("/kaggle/input"):
            if filename in files:
                return os.path.join(root, filename)
    raise FileNotFoundError(f"Could not find {filename}. Please attach your dataset to this notebook.")

TRAIN_FILE = find_dataset_file("train.jsonl")
print(f"[Found Dataset] Using: {TRAIN_FILE}")

In [ ]:
# Cell 3: Load Model, Tokenizer & Setup LoRA
import torch
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

MAX_SEQ_LENGTH = 2048
MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"

print("Loading base model Qwen 2.5 7B Instruct...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

# Apply Qwen 2.5 ChatML format
tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen-2.5",
    mapping={"role": "role", "content": "content", "user": "user", "assistant": "assistant"},
)

# Configure LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print("Model & LoRA initialized successfully!")

In [ ]:
# Cell 4: Prepare Dataset & Train
def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in convos
    ]
    return {"text": texts}

print("Formatting dataset with ChatML...")
dataset = load_dataset("json", data_files={"train": TRAIN_FILE})["train"]
dataset = dataset.map(formatting_prompts_func, batched=True)

training_args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    num_train_epochs=2,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=3407,
    output_dir="/kaggle/working/outputs",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
)

print("\n--- Starting Fine-Tuning ---")
trainer_stats = trainer.train()
print("--- Fine-Tuning Complete! ---")

In [ ]:
# Cell 5: Quick Tone Verification Test
FastLanguageModel.for_inference(model)
test_messages = [
    {"role": "system", "content": "You are chatting on WhatsApp as Ammar akbar."},
    {"role": "user", "content": "kya scene hai bhai? free ho?"}
]
inputs = tokenizer.apply_chat_template(test_messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = model.generate(input_ids=inputs, max_new_tokens=64, temperature=0.7)
reply = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

print("\n===========================")
print("Test Response from Twin:")
print(reply)
print("===========================\n")

In [ ]:
# Cell 6: Export LoRA Adapter as GGUF & Trigger Fast Download
import os
from IPython.display import FileLink, display, HTML

# Bypass disk preflight check since LoRA adapter is only ~50MB
os.environ["UNSLOTH_DISK_PREFLIGHT"] = "0"

LORA_GGUF_DIR = "/kaggle/working/qwen_whatsapp_lora_gguf"
os.makedirs(LORA_GGUF_DIR, exist_ok=True)

print("Exporting LoRA adapter to GGUF format (save_method='lora')...")
model.save_pretrained_gguf(
    LORA_GGUF_DIR,
    tokenizer,
    save_method="lora",
)
print("LoRA GGUF export complete!")

# Also save standard PEFT adapter as backup (~50MB)
model.save_pretrained("/kaggle/working/qwen_whatsapp_lora")
tokenizer.save_pretrained("/kaggle/working/qwen_whatsapp_lora")

# Find generated .gguf adapter file
gguf_file = None
for root, _, files in os.walk(LORA_GGUF_DIR):
    for f in files:
        if f.endswith(".gguf"):
            gguf_file = os.path.join(root, f)
            break

if gguf_file and os.path.exists(gguf_file):
    size_mb = os.path.getsize(gguf_file) / (1024**2)
    rel_path = os.path.relpath(gguf_file, "/kaggle/working")
    print(f"\n[Ready] LoRA GGUF Adapter created: {gguf_file} ({size_mb:.1f} MB)")
    print("Click below to download your fine-tuned adapter:")
    display(FileLink(rel_path))
    display(HTML(f'''
        <div style="margin-top: 15px; padding: 15px; background: #e8f5e9; border: 2px solid #4caf50; border-radius: 8px;">
            <h3 style="color: #2e7d32; margin: 0 0 10px 0;">LoRA Adapter Ready</h3>
            <p>Your fine-tuned adapter <b>{os.path.basename(gguf_file)}</b> is ready ({size_mb:.1f} MB).</p>
            <a href="{rel_path}" download style="display: inline-block; padding: 10px 20px; background: #2e7d32; color: white; text-decoration: none; border-radius: 5px; font-weight: bold;">Download LoRA GGUF</a>
        </div>
    '''))
else:
    print("Look in /kaggle/working/qwen_whatsapp_lora_gguf in the right panel to download the .gguf file.")